In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import pyaging as pya

In [3]:
path = "D:/bioTest/attack/"
pheno = pd.read_excel(f"{path}controls.xlsx", index_col=0)
betas = pd.read_pickle(f"{path}betas.pkl")

feats_pheno = ['Age', 'Sex', 'Tissue']
pheno = pheno[feats_pheno]

df_clocks = pd.merge(pheno, betas, left_index=True, right_index=True)
df_clocks['Female'] = (df_clocks['Sex'] == 'F').astype(int)

In [ ]:
clocks = ["horvath2013"]

adata = pya.pp.df_to_adata(df_clocks, metadata_cols=['Sex', 'Tissue'], imputer_strategy='knn', verbose=True)
pya.pred.predict_age(adata=adata, dir=path, clock_names=clocks, verbose=True)
results = pd.merge(pheno.loc[:, feats_pheno], adata.obs[clocks], left_index=True, right_index=True)
results.to_excel(f"{path}/clock.xlsx")

In [ ]:
logger = pya.logger.Logger('test_logger')
device = 'cpu'
dir = 'pyaging_data'
indent_level = 1

clock = pya.pred.load_clock(clocks[0], device, dir, logger, indent_level=indent_level)
clock_features = clock.features
clock_reference_values = clock.reference_values

In [6]:
common_cpgs = list(set(clock_features).intersection(betas.columns))
differ_cpgs = list(set(clock_features).difference(betas.columns))
missing_indices = [clock_features.index(curr_cpg) for curr_cpg in differ_cpgs]
missing_references = [clock_reference_values[ref] for ref in missing_indices]
df = betas.loc[:, common_cpgs]
df[differ_cpgs] = pd.DataFrame([missing_references], index=df.index)
df = pd.merge(df, results, left_index=True, right_index=True)

In [37]:
import torch
from art.estimators.regression.pytorch import PyTorchRegressor

feats = clock_features
ids_feat = list(range(len(feats)))
col_trgt = 'Age'
col_pred = 'horvath2013'

df["Error"] = df["horvath2013"] - df["Age"]
df["|Error|"] = df["Error"].abs()
df['Data'] = 'Real'
df['Eps'] = 'Origin'

model = clock
model.base_model.linear.weight = torch.nn.parameter.Parameter(model.base_model.linear.weight.to(torch.float32))
model.base_model.linear.bias = torch.nn.parameter.Parameter(model.base_model.linear.bias.to(torch.float32))

def predict_func_regression(X):
    model.produce_probabilities = True
    batch = {
        'all': torch.from_numpy(np.float32(X[:, ids_feat])),
        'continuous': torch.from_numpy(np.float32(X[:, ids_feat])),
        'categorical': torch.from_numpy(np.int32(X[:, []])),
    }
    tmp = model(batch)
    return tmp.cpu().detach().numpy()

art_regressor = PyTorchRegressor(
    model=model,
    loss=torch.nn.L1Loss(),
    input_shape=[len(feats)],
    use_amp=False,
    opt_level="O1",
    loss_scale="dynamic",
    channels_first=True,
    clip_values=None,
    preprocessing_defences=None,
    postprocessing_defences=None,
    preprocessing=(0.0, 1.0),
    device_type="cpu",
)

In [8]:
import pathlib
from scipy.stats import iqr
from art.attacks.evasion import BasicIterativeMethod
from metrics import get_reg_metrics
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
tensor_for_model = torch.from_numpy(np.float32(df.loc[:, feats].values))
y_test_pred1 = model(tensor_for_model)
y_test_pred2 = clock.base_model(tensor_for_model)
y_test_pred3 = clock(tensor_for_model)

In [29]:
olol = art_regressor.predict(np.float32(df.loc[:, feats].values))

In [ ]:
print(art_regressor.model.linear.weight.dtype)
print(art_regressor.model.linear.bias.dtype)

In [ ]:
model.linear.weight = torch.nn.parameter.Parameter(model.linear.weight.to(torch.float32))
model.linear.weight.dtype

In [40]:
epsilons = sorted(list(set.union(
    set(np.linspace(0.1, 1.0, 10)), 
    set(np.linspace(0.01, 0.1, 10)),
)))
df_eps = pd.DataFrame(index=epsilons)

for eps_raw in epsilons:

    eps = np.array([eps_raw * iqr(df.loc[:, feat].values) for feat in feats])
    eps_step = np.array([0.2 * eps_raw * iqr(df.loc[:, feat].values) + 1e-6 for feat in feats])

    attacks = {
        'BasicIterative': BasicIterativeMethod(
            estimator=art_regressor,
            eps=eps,
            eps_step=eps_step,
            max_iter=100,
            targeted=False,
            batch_size=512,
            verbose=True
        )
    }

    for attack_name, attack in attacks.items():
        path_curr = f"{path}/Evasion/{attack_name}/eps_{eps_raw:0.4f}"
        pathlib.Path(f"{path_curr}").mkdir(parents=True, exist_ok=True)

        X_adv = attack.generate(df.loc[:, feats].values.astype(np.float32))
        
        df_adv = df.loc[:, ['Age']].copy()
        df_adv.loc[:, feats] = X_adv
        df_adv["horvath2013"] = model(torch.from_numpy(np.float32(df_adv.loc[:, feats].values))).cpu().detach().numpy().ravel()
        df_adv["Error"] = df_adv["horvath2013"] - df_adv["Age"]
        df_adv["abs(Error)"] = df_adv["Error"].abs()
        df_adv.loc[:, "Error Origin"] = df.loc[:, "horvath2013"] - df.loc[:, "Age"]
        df_adv.loc[:, "Error Attack"] = df_adv.loc[:, "horvath2013"] - df_adv.loc[:, "Age"]
        df_adv['Error Diff'] = df_adv['Error Attack'] - df_adv['Error Origin']
        df_adv['abs(Error Diff)'] = df_adv['Error Diff'].abs()
            
        df_adv.to_excel(f"{path_curr}/df.xlsx", index_label='sample_id')

        metrics = get_reg_metrics()
        metrics_cols = [f"{m}" for m in metrics]
        df_metrics = pd.DataFrame(index=metrics_cols)
        for m in metrics:
            m_val = float(metrics[m][0](torch.from_numpy(np.float32(df.loc[:, "horvath2013"].values)), torch.from_numpy(np.float32(df.loc[:, "Age"].values))).numpy())
            df_metrics.at[f"{m}", 'Origin'] = m_val
            metrics[m][0].reset()
            m_val = float(metrics[m][0](torch.from_numpy(np.float32(df_adv.loc[:, "horvath2013"].values)), torch.from_numpy(np.float32(df.loc[:, "Age"].values))).numpy())
            df_metrics.at[f"{m}", 'Attack'] = m_val
            metrics[m][0].reset()
        df_metrics.to_excel(f"{path_curr}/metrics.xlsx", index_label='Metrics')
    
        df_eps.loc[eps_raw, f"Origin_MAE"] = df_metrics.at[f'mean_absolute_error', 'Origin']
        df_eps.loc[eps_raw, f"{attack_name}_MAE"] = df_metrics.at[f'mean_absolute_error', 'Attack']

df_eps.to_excel(f"{path}/Evasion/df_eps.xlsx", index_label='eps')

df_fig = df_eps.copy()
df_fig['Eps'] = df_fig.index.values
df_fig = df_fig.melt(id_vars="Eps", var_name='Method', value_name="MAE")
fig = plt.figure()
sns.set_theme(style='whitegrid', font_scale=1)
lines = sns.lineplot(
    data=df_fig,
    x='Eps',
    y="MAE",
    hue=f"Method",
    style=f"Method",
    markers=True,
    dashes=False,
)
plt.xscale('log')
lines.set_xlabel(r'$\epsilon$')
x_min = 0.009
x_max = 1.05
mae_basic = df_eps.at[0.01, f"Origin_MAE"]
lines.set_xlim(x_min, x_max)
plt.gca().plot(
    [x_min, x_max],
    [mae_basic, mae_basic],
    color='k',
    linestyle='dashed',
    linewidth=1
)
plt.savefig(f"{path}/Evasion/line_mae_vs_eps.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path}/Evasion/line_mae_vs_eps.pdf", bbox_inches='tight')
plt.close(fig)


PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_11356\2338639558.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_11356\2338639558.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

...

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_11356\2338639558.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_11356\2338639558.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_11356\2338639558.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

In [ ]:
from attack.sim.ft_transformer import WDFTTransformerModel

pathsim = "E:/YandexDisk/pydnameth/datasets/GPL21145/GSEUNN"
path_model = f"{pathsim}/data/immuno/models/SImAge"
dfsim = pd.read_excel(f"{pathsim}/data/immuno/models/SImAge/data.xlsx", index_col='sample_id')
featssim = pd.read_excel(f"{pathsim}/data/immuno/models/SImAge/feats_con_top10.xlsx", index_col=0).index.values
ids_feat = list(range(len(featssim)))
col_trgt = 'Age'
col_pred = 'SImAge'

dfsim_preds = pd.read_excel(f"{pathsim}/data/immuno/models/SImAge/results/predictions.xlsx", index_col=0)
ids_trn = dfsim_preds.index[dfsim_preds['fold_0002'] == 'trn'].values
ids_val = dfsim_preds.index[dfsim_preds['fold_0002'] == 'val'].values
ids_tst = dfsim_preds.index[dfsim_preds['fold_0002'] == 'tst_ctrl_central'].values
ids_all = dfsim_preds.index[dfsim_preds['fold_0002'].isin(['trn', 'val', 'tst_ctrl_central'])].values
ids_trn_val = dfsim_preds.index[dfsim_preds['fold_0002'].isin(['trn', 'val'])].values
ids_dict = {
    'all': ids_all,
    'trn_val': ids_trn_val,
    'tst': ids_tst
}

dfsim = dfsim.loc[ids_all, :]
dfsim["SImAge Error"] = dfsim["SImAge"] - dfsim["Age"]
dfsim["abs(SImAge Error)"] = dfsim["SImAge Error"].abs()
dfsim['Data'] = 'Real'
dfsim['Eps'] = 'Origin'

modelsim = WDFTTransformerModel.load_from_checkpoint(checkpoint_path=f"{pathsim}/data/immuno/models/SImAge/best_fold_0002.ckpt")
modelsim.eval()
modelsim.freeze()

def predict_func_regression(X):
    modelsim.produce_probabilities = True
    batch = {
        'all': torch.from_numpy(np.float32(X[:, ids_feat])),
        'continuous': torch.from_numpy(np.float32(X[:, ids_feat])),
        'categorical': torch.from_numpy(np.int32(X[:, []])),
    }
    tmp = modelsim(batch)
    return tmp.cpu().detach().numpy()

art_regressorsim = PyTorchRegressor(
    model=modelsim,
    loss=modelsim.loss_fn,
    input_shape=[len(featssim)],
    optimizer=torch.optim.Adam(
        params=modelsim.parameters(),
        lr=modelsim.hparams.optimizer_lr,
        weight_decay=modelsim.hparams.optimizer_weight_decay
    ),
    use_amp=False,
    opt_level="O1",
    loss_scale="dynamic",
    channels_first=True,
    clip_values=None,
    preprocessing_defences=None,
    postprocessing_defences=None,
    preprocessing=(0.0, 1.0),
    device_type="cpu",
)

In [ ]:
ololsim = art_regressorsim.predict(dfsim.loc[:, featssim].values)
ololsim

In [ ]:
epsilons = sorted(list(set.union(
    set(np.linspace(0.1, 1.0, 10)), 
    set(np.linspace(0.01, 0.1, 10)),
)))
df_eps = pd.DataFrame(index=epsilons)

for eps_raw in epsilons:

    eps = np.array([eps_raw * iqr(dfsim.loc[:, feat].values) for feat in featssim])
    eps_step = np.array([0.2 * eps_raw * iqr(dfsim.loc[:, feat].values) for feat in featssim])

    attacks = {
        'BasicIterative': BasicIterativeMethod(
            estimator=art_regressorsim,
            eps=eps,
            eps_step=eps_step,
            max_iter=100,
            targeted=False,
            batch_size=512,
            verbose=True
        ),
    }

    for attack_name, attack in attacks.items():

        X_adv = attack.generate(np.float32(dfsim.loc[:, featssim].values))
        
        df_adv = dfsim.loc[:, ['Age']].copy()
        df_adv.loc[:, featssim] = X_adv
        df_adv["SImAge"] = model(torch.from_numpy(np.float32(df_adv.loc[:, featssim].values))).cpu().detach().numpy().ravel()
        df_adv["SImAge Error"] = df_adv["SImAge"] - df_adv["Age"]
        df_adv["abs(SImAge Error)"] = df_adv["SImAge Error"].abs()
        df_adv.loc[:, "Error Origin"] = dfsim.loc[:, "SImAge"] - dfsim.loc[:, "Age"]
        df_adv.loc[:, "Error Attack"] = df_adv.loc[:, "SImAge"] - df_adv.loc[:, "Age"]
        df_adv['Error Diff'] = df_adv['Error Attack'] - df_adv['Error Origin']
        df_adv['abs(Error Diff)'] = df_adv['Error Diff'].abs()

        metrics = get_reg_metrics()
        metrics_cols = [f"{m}_{p}" for m in metrics for p in ids_dict]
        df_metrics = pd.DataFrame(index=metrics_cols)
        for p, ids_part in ids_dict.items():
            for m in metrics:
                m_val = float(metrics[m][0](torch.from_numpy(np.float32(dfsim.loc[ids_part, "SImAge"].values)), torch.from_numpy(np.float32(dfsim.loc[ids_part, "Age"].values))).numpy())
                df_metrics.at[f"{m}_{p}", 'Origin'] = m_val
                metrics[m][0].reset()
                m_val = float(metrics[m][0](torch.from_numpy(np.float32(df_adv.loc[ids_part, "SImAge"].values)), torch.from_numpy(np.float32(dfsim.loc[ids_part, "Age"].values))).numpy())
                df_metrics.at[f"{m}_{p}", 'Attack'] = m_val
                metrics[m][0].reset()
        
        for p in ids_dict:
            if attack_name == 'MomentumIterative':
                df_eps.loc[eps_raw, f"Origin_MAE_{p}"] = df_metrics.at[f'mean_absolute_error_{p}', 'Origin']
            df_eps.loc[eps_raw, f"{attack_name}_MAE_{p}"] = df_metrics.at[f'mean_absolute_error_{p}', 'Attack']